# verification_summarization

`POST /analyses/verification_summarization/`

Converts raw L3 verification rule failures into plain-language flag messages for the
field team. Text in, text out -- it never touches an image, which is why it sits under
`/analyses` rather than `/computer_vision`.

**Needs:** the API key for whichever provider `configs/config_cv_verification_summarization.yaml`
names (`model: "claude"` by default, so `ANTHROPIC_API_KEY`).

> **Path differs from the deployed service:** live is `/verification_summarization/`,
> here it is `/analyses/verification_summarization/`.

In [ ]:
import json
import os
import sys

import requests
from dotenv import load_dotenv

# Run against a locally started service:  python3 src/main.py pipeline pipeline_api
sys.path.insert(0, os.path.abspath(".."))
load_dotenv(os.path.join("..", ".env"))

BASE_URL = os.getenv("VT_API_BASE_URL", "http://localhost:8000")
TOKEN = os.getenv("API_ENDPOINT_TOKEN")
HEADERS = {"Token": TOKEN}

assert TOKEN, "API_ENDPOINT_TOKEN missing -- copy env_template to .env and fill it in"
print(f"target: {BASE_URL}")

In [ ]:
# Confirm the service is up before sending anything to the route.
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5)
    print(health.status_code, health.json())
except requests.exceptions.ConnectionError:
    print("Service is not running. Start it with:\n"
          "    python3 src/main.py pipeline pipeline_api")

## Request

Send the rules that failed. `comments` are included in the prompt, so field-team context reaches the model.

In [ ]:
payload = {
    "rules": [
        {
            "rule_public_id": "rule_abc123",
            "name": "Photo missing meterstick",
            "status": "failed",
            "failure_reason": {"detail": "no meterstick detected in 4 of 12 photos"},
            "flagged": True,
            "comments": [
                {"source": "field", "actor_id": "u1",
                 "message": "stick was out of frame on the slope", "created_at": "2026-03-16"}
            ],
            "category": "evidence",
            "group": "photo",
        },
    ]
}

response = requests.post(f"{BASE_URL}/analyses/verification_summarization/",
                         json=payload, headers=HEADERS, timeout=120)

In [ ]:
print(response.status_code)
if response.ok:
    body = response.json()
    print(json.dumps(body, indent=2)[:2000])
else:
    print(response.text[:1000])

## Suggested flags

In [ ]:
if response.ok:
    for flag in response.json()["suggested_flags"]:
        print(f"[{flag['rule_public_id']}]\n  {flag['message']}\n")

## Error cases

The model is asked for a JSON array. Malformed output surfaces as a **500** rather than a partial response, so a bad generation is never mistaken for 'no flags'.

In [ ]:
for label, body in [
    ("empty rule list ", {"rules": []}),
    ("missing fields  ", {"rules": [{"name": "incomplete"}]}),
]:
    r = requests.post(f"{BASE_URL}/analyses/verification_summarization/",
                      json=body, headers=HEADERS, timeout=60)
    print(f"{label} -> {r.status_code}")